# Course PR Reviewer

Reviews pull requests on [`ed-donner/llm_engineering`](https://github.com/ed-donner/llm_engineering/pulls) for **any week**.

**Flow:** PR number → GitHub API (title, body, files, comments, truncated diff) → OpenAI review → optionally post a comment.

Extras:
1. **Display** existing PR conversation comments
2. **Post** a comment (needs `GITHUB_TOKEN` with permission on the repo; off by default)

Works for `community-contributions/` and `weekN/community-contributions/`. Day-1 sized: prompts + messages + API calls. No Selenium.


In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
# setup

load_dotenv(override=True)
openai = OpenAI()

REPO = "ed-donner/llm_engineering"
MODEL = "gpt-4.1-mini"
MAX_DIFF_CHARS = 12_000  # keep prompts small; notebooks can be huge
GITHUB_API = "https://api.github.com"

headers = {
    "Accept": "application/vnd.github+json",
    "User-Agent": "llm-engineering-day1-pr-reviewer",
}
# Make sure to set the GITHUB_TOKEN environment variable
token = os.getenv("GITHUB_TOKEN")
if token:
    headers["Authorization"] = f"Bearer {token}"


In [ ]:
# fetch PR metadata + truncated diff from the public GitHub API

def fetch_pr(pr_number: int) -> dict:
    base = f"{GITHUB_API}/repos/{REPO}/pulls/{pr_number}"
    meta = requests.get(base, headers=headers, timeout=30)
    meta.raise_for_status()
    pr = meta.json()

    files_resp = requests.get(f"{base}/files", headers=headers, params={"per_page": 100}, timeout=30)
    files_resp.raise_for_status()
    files = [f["filename"] for f in files_resp.json()]

    diff_headers = {**headers, "Accept": "application/vnd.github.diff"}
    diff_resp = requests.get(base, headers=diff_headers, timeout=60)
    diff_resp.raise_for_status()
    diff = diff_resp.text
    truncated = False
    if len(diff) > MAX_DIFF_CHARS:
        diff = diff[:MAX_DIFF_CHARS] + "\n\n...[diff truncated]..."
        truncated = True

    return {
        "number": pr["number"],
        "title": pr["title"],
        "body": pr.get("body") or "(no description)",
        "author": (pr.get("user") or {}).get("login", "unknown"),
        "url": pr.get("html_url"),
        "additions": pr.get("additions"),
        "deletions": pr.get("deletions"),
        "changed_files": pr.get("changed_files"),
        "files": files,
        "diff": diff,
        "diff_truncated": truncated,
    }


def fetch_pr_comments(pr_number: int, limit: int = 20) -> list[dict]:
    """Conversation comments on the PR (issue comments endpoint)."""
    url = f"{GITHUB_API}/repos/{REPO}/issues/{pr_number}/comments"
    resp = requests.get(url, headers=headers, params={"per_page": limit}, timeout=30)
    resp.raise_for_status()
    comments = []
    for c in resp.json():
        comments.append({
            "user": (c.get("user") or {}).get("login", "unknown"),
            "created_at": c.get("created_at"),
            "body": c.get("body") or "",
            "url": c.get("html_url"),
        })
    return comments


def display_comments(pr_number: int, limit: int = 20):
    comments = fetch_pr_comments(pr_number, limit=limit)
    if not comments:
        display(Markdown(f"_No conversation comments yet on PR #{pr_number}._"))
        return comments

    parts = [f"### Existing comments on PR #{pr_number} ({len(comments)})", ""]
    for c in comments:
        parts.append(f"**@{c['user']}** ({c['created_at']})")
        parts.append("")
        parts.append(c["body"].strip())
        parts.append("")
        parts.append("---")
        parts.append("")
    display(Markdown("\n".join(parts)))
    return comments


def post_pr_comment(pr_number: int, body: str) -> dict:
    """Post a conversation comment. Requires GITHUB_TOKEN with write access."""
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN in .env before posting comments.")
    if not body or not body.strip():
        raise ValueError("Comment body is empty.")

    url = f"{GITHUB_API}/repos/{REPO}/issues/{pr_number}/comments"
    resp = requests.post(url, headers=headers, json={"body": body}, timeout=30)
    if resp.status_code == 403:
        raise PermissionError(
            "GitHub returned 403 — your token likely cannot comment on this repo. "
            "Copy the suggested review comment and paste it manually instead."
        )
    resp.raise_for_status()
    data = resp.json()
    print(f"Posted comment: {data.get('html_url')}")
    return data


In [ ]:
# system prompt — Ed's contribution guidelines (any week)

SYSTEM_PROMPT = """
You are a careful PR reviewer for the open-source course repo ed-donner/llm_engineering.

Review community contributions against these rules (apply to any week, not only week1):
1. Changes should live under `community-contributions/` or `weekN/community-contributions/` (e.g. week1, week2, …) unless discussed otherwise.
2. Notebook outputs should be cleared.
3. Keep it small: ideally under ~2,000 lines total, not too many files, avoid huge dumps.
4. No secrets, .env files, credentials, or API keys.
5. Avoid unnecessary test clutter, overly wordy READMEs, emoji spam, or generic LLM artifacts.
6. Prefer useful, runnable student work over drive-by noise.
7. Do not edit core course lab files (e.g. `day*.ipynb`, `scraper.py`, shared course modules) unless that is explicitly the point of the PR.

Be practical and fair. If the diff was truncated, say so and judge only what you can see.
Use existing PR comments as context; do not repeat them unless needed.
Name the actual paths/files you saw (week2, week5, etc.) — do not assume week1.

Respond in markdown with exactly these sections:
### Verdict
Approve | Comment | Request changes

### Summary
2-4 short bullets

### Guideline checks
One line per rule: Pass or Fail — brief reason

### Risks / questions
Short bullets (or "None")

### Suggested review comment
This section is posted to GitHub. Format it EXACTLY like this template (fill in real findings; omit Pass rows):

```
## PR review

**Verdict:** Request changes

### Checklist
| Guideline | Result |
| --- | --- |
| Under `community-contributions/` (or `weekN/community-contributions/`) | Fail — files in `week2/` root |
| Notebook outputs cleared | Fail — outputs present |
| Small / focused diff | Fail — noisy extra files |
| No secrets / credentials | Fail — `credentials.json` |
| No emoji/wordy README clutter | Fail — ... |
| No test clutter | Fail — `random_tests/` |
| No unrelated lab-file edits | Fail — edited `day1.ipynb` / `scraper.py` |

### Please fix
1. Move work under `weekN/community-contributions/<yourname>/` (use the correct week)
2. Clear notebook outputs
3. Remove secrets, emoji README, and test clutter
4. Do not edit core course lab files in a community contribution PR

Thanks!
```

Rules for the suggested comment:
- Use markdown headings + a checklist table + a short numbered fix list
- Keep it under ~120 words when possible
- No long paragraphs, no lectures, no emoji
- Be direct and kind
- Reference the real week/paths from the diff
- If Approve, use a short thanks + 2-3 what-went-well bullets instead of Fail rows
""".strip()


def messages_for(pr: dict, comments: list[dict] | None = None) -> list[dict]:
    file_list = "\n".join(f"- {f}" for f in pr["files"]) or "(none listed)"
    if comments:
        comment_block = "\n\n".join(
            f"@{c['user']} ({c['created_at']}):\n{c['body']}" for c in comments
        )
    else:
        comment_block = "(no comments)"

    user = f"""
Review this pull request.

PR: #{pr['number']} — {pr['title']}
Author: {pr['author']}
URL: {pr['url']}
Stats: +{pr['additions']} / -{pr['deletions']} across {pr['changed_files']} files
Diff truncated: {pr['diff_truncated']}

## Description
{pr['body']}

## Existing comments
{comment_block}

## Files changed
{file_list}

## Diff
```diff
{pr['diff']}
```
""".strip()
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


def review_pr(pr_number: int) -> str:
    pr = fetch_pr(pr_number)
    comments = fetch_pr_comments(pr_number)
    response = openai.chat.completions.create(model=MODEL, messages=messages_for(pr, comments))
    return response.choices[0].message.content


def display_review(pr_number: int) -> str:
    review = review_pr(pr_number)
    display(Markdown(review))
    return review


def extract_suggested_comment(review_markdown: str) -> str:
    """Pull the 'Suggested review comment' section for optional posting.

    Important: do NOT stop at '###' — the comment template itself uses ### headings
    (Checklist / Please fix). Truncating there posts a blank/broken GitHub comment.
    """
    marker = "Suggested review comment"
    lower = review_markdown.lower()
    idx = lower.find(marker.lower())
    if idx == -1:
        return review_markdown.strip()

    section = review_markdown[idx + len(marker):].lstrip(" :\n*-")

    # Unwrap a single fenced code block if the model wrapped the comment
    if section.startswith("```"):
        lines = section.splitlines()
        # drop opening fence (``` or ```markdown)
        lines = lines[1:]
        # drop closing fence if present
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        section = "\n".join(lines).strip()

    section = section.strip()
    if len(section) < 20:
        raise ValueError(
            "Extracted suggested comment looks empty/too short. "
            "Not posting — inspect `review` output and try again."
        )
    return section


In [ ]:
# pick any open PR from https://github.com/ed-donner/llm_engineering/pulls

PR_NUMBER = 3663  # change me

pr = fetch_pr(PR_NUMBER)
print(f"#{pr['number']} {pr['title']}")
print(f"by {pr['author']} | +{pr['additions']}/-{pr['deletions']} | {pr['changed_files']} files")
print(pr['url'])
print("files:")
for f in pr['files'][:20]:
    print(" ", f)
if len(pr['files']) > 20:
    print(f"  ... +{len(pr['files']) - 20} more")


In [ ]:
# 1) display existing PR conversation comments
comments = display_comments(PR_NUMBER)
print(f"{len(comments)} comment(s) loaded")


In [ ]:
# 2) generate LLM review (uses comments as context)
review = display_review(PR_NUMBER)
suggested = extract_suggested_comment(review)
print("\n--- Suggested comment to post ---")
print(suggested)


In [ ]:
# 3) optionally post the suggested comment to GitHub
# Requires GITHUB_TOKEN with permission to comment on ed-donner/llm_engineering.
# If you get 403, copy `suggested` and paste manually on the PR.
POST_COMMENT = True  # set True only if you have GITHUB_TOKEN write access and want to post

if POST_COMMENT:
    post_pr_comment(PR_NUMBER, suggested)
else:
    print("POST_COMMENT is False — not posting. Set POST_COMMENT = True to post.")
